# Delta Ops — Partitioning & Z-ORDER

Analyze partition column cardinality, build **`fact_sales_partitioned`** by `order_year_month`, run **OPTIMIZE ZORDER BY**, and capture a partition-pruning **explain** plan.

In [ ]:
import sys
import importlib

notebook_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
REPO_ROOT = notebook_path.split("/notebooks/")[0]
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

import src.delta_ops.partitioning as part_module
importlib.reload(part_module)

from src.delta_ops.partitioning import PartitionStrategyConfig, run_partition_zorder_strategy
from src.ingestion.idempotent_loader import save_run_report_to_volume

config = PartitionStrategyConfig()
print("Loaded from:", part_module.__file__)

In [ ]:
report = run_partition_zorder_strategy(spark, config)
print("Cardinality:", report["cardinality_analysis"])
print("Explain snippet:")
print(report["partition_prune_explain_snippet"])

In [ ]:
import json

print(json.dumps(report, indent=2, default=str))
try:
    save_run_report_to_volume(
        report,
        dbutils,
        "/Volumes/globalmart/metadata/run_reports/delta_partition_zorder.json",
    )
except Exception as exc:
    print("Volume save skipped:", exc)